# Indian Multi-City Bus ETA — Colab A100 Deep Residual + Corridor Embedding Model (DeeprETA Style)

Trains a heavyweight **Deep Residual Neural Network with Learned Entity Embeddings** (`city`, `bus_class`, `phys_link_id` corridor hash, `route_id` hash) + **Multi-Quantile Pinball Loss (P10 / P50 / P90)** + **Conformalized Quantile Regression (CQR)** on your processed multi-city Parquet datasets (`delhi`, `mtc`, `bmtc`, `mumbai`, `ncr`), including `pol.reclaimchennai.city` weather/rainfall + headway + shared-corridor probes.

**Recommended Runtime:** Google Colab **A100 GPU (40GB/80GB)**.

In [ ]:
!pip -q install polars pyarrow torch lightgbm catboost scikit-learn
import torch, polars as pl, numpy as np, json, math, time
from pathlib import Path
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Mount Google Drive and point DATA_DIR to your uploaded ETA/data/processed folder
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/ETA/processed')
OUT_DIR = Path('/content/drive/MyDrive/ETA/models/deep_eta_a100')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Parquet files:', sorted([p.name for p in DATA_DIR.glob('*_segments.parquet')]))

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

HASH_CORR = 65536
HASH_ROUTE = 16384

NUM_COLS = [
    'stop_seq', 'route_num_stops', 'full_seg_len_m', 'seg_dist_m', 'sinuosity', 'mid_lat', 'mid_lon',
    'hour_ist', 'min_of_day', 'dow', 'is_weekend', 'sin_tod', 'cos_tod', 'sin_dow', 'cos_dow',
    'veh_pace_ewma', 'trip_elapsed_s', 'trip_dist_m', 'veh_recent_v', 'raw_k',
    'seg_live_v_15m', 'seg_live_n_15m', 'corr_live_v_5m', 'corr_live_v_15m', 'corr_live_v_30m',
    'corr_live_n_15m', 'corr_speed_cv_30m', 'corr_stopgo_30m', 'up_live_v_15m', 'dn_live_v_15m',
    'city_live_v_15m', 'lead_headway_s', 'corr_lead_headway_s',
    'temp_c', 'humidity', 'precip_mm', 'precip_3h_mm', 'precip_6h_mm', 'precip_24h_mm',
    'is_raining', 'is_heavy_rain', 'wind_kph', 'gust_kph', 'cloud_cover', 'vis_km', 'pm25', 'aqi'
]

class ResBlock(nn.Module):
    def __init__(self, d: int, dropout: float = 0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(d), nn.SiLU(), nn.Dropout(dropout), nn.Linear(d, d),
            nn.LayerNorm(d), nn.SiLU(), nn.Dropout(dropout), nn.Linear(d, d),
        )
    def forward(self, x):
        return x + self.block(x)

class DeepIndianBusETA(nn.Module):
    """DeeprETA-inspired multi-city residual network with learned corridor & route embeddings."""
    def __init__(self, n_num: int, n_cities: int = 8, n_classes: int = 128, d_model: int = 384, n_blocks: int = 4):
        super().__init__()
        self.emb_city = nn.Embedding(n_cities, 16)
        self.emb_cls = nn.Embedding(n_classes, 16)
        self.emb_corr = nn.Embedding(HASH_CORR, 48)
        self.emb_route = nn.Embedding(HASH_ROUTE, 32)
        in_dim = n_num + 16 + 16 + 48 + 32
        self.in_proj = nn.Linear(in_dim, d_model)
        self.blocks = nn.ModuleList([ResBlock(d_model, dropout=0.1) for _ in range(n_blocks)])
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.SiLU(), nn.Linear(d_model, 3))  # P10, P50, P90

    def forward(self, x_num, c_city, c_cls, c_corr, c_route):
        h = torch.cat([
            x_num,
            self.emb_city(c_city),
            self.emb_cls(c_cls),
            self.emb_corr(c_corr),
            self.emb_route(c_route),
        ], dim=-1)
        h = self.in_proj(h)
        for b in self.blocks:
            h = b(h)
        return self.head(h)

def pinball_loss(preds, target, qs=(0.10, 0.50, 0.90)):
    loss = 0.0
    for i, q in enumerate(qs):
        e = target - preds[:, i]
        loss = loss + torch.maximum(q * e, (q - 1.0) * e).mean()
    return loss / len(qs)

print('Model defined successfully.')

In [ ]:
# Load all cities, strict temporal split (train / val / test), train with bf16 mixed precision on A100
cities = ['delhi', 'mtc', 'bmtc', 'mumbai', 'ncr']
frames = []
for ci, city in enumerate(cities):
    p = DATA_DIR / f'{city}_segments.parquet'
    if not p.exists():
        continue
    df = pl.read_parquet(p).filter((pl.col('eff_speed_ms') >= 0.4) & (pl.col('eff_speed_ms') <= 22.0))
    days = sorted(df['day'].unique().to_list())
    if len(days) < 3:
        continue
    test_d, val_d = days[-2], days[-3]  # avoid partial last day
    df = df.with_columns([
        pl.lit(ci, dtype=pl.Int64).alias('city_id'),
        pl.when(pl.col('day') == test_d).then(2).when(pl.col('day') == val_d).then(1).otherwise(0).alias('split_id'),
        (pl.col('phys_link_id').hash(seed=42) % HASH_CORR).cast(pl.Int64).alias('corr_id'),
        (pl.col('route_id').hash(seed=42) % HASH_ROUTE).cast(pl.Int64).alias('route_hash'),
        (pl.col('bus_class').hash(seed=42) % 128).cast(pl.Int64).alias('cls_id'),
        pl.col('eff_speed_ms').log().alias('target_log_v'),
    ])
    for c in NUM_COLS:
        if c not in df.columns:
            df = df.with_columns(pl.lit(0.0, dtype=pl.Float32).alias(c))
        else:
            df = df.with_columns(pl.col(c).cast(pl.Float32).fill_null(0.0).fill_nan(0.0))
    frames.append(df.select(NUM_COLS + ['city_id', 'cls_id', 'corr_id', 'route_hash', 'split_id', 'target_log_v', 'seg_dist_m', 'dur_s']))

full = pl.concat(frames, how='vertical_relaxed')
print(f'Total rows loaded: {len(full):,}')

tr = full.filter(pl.col('split_id') == 0)
va = full.filter(pl.col('split_id') == 1)
te = full.filter(pl.col('split_id') == 2)

# Standardize numeric features using train stats only
X_tr = tr.select(NUM_COLS).to_numpy().astype(np.float32)
mu, sd = X_tr.mean(axis=0), X_tr.std(axis=0) + 1e-6
X_tr = (X_tr - mu) / sd
X_va = (va.select(NUM_COLS).to_numpy().astype(np.float32) - mu) / sd
X_te = (te.select(NUM_COLS).to_numpy().astype(np.float32) - mu) / sd

def make_loader(df_s, X_s, bs=8192, shuffle=False):
    ds = TensorDataset(
        torch.from_numpy(X_s),
        torch.from_numpy(df_s['city_id'].to_numpy()),
        torch.from_numpy(df_s['cls_id'].to_numpy()),
        torch.from_numpy(df_s['corr_id'].to_numpy()),
        torch.from_numpy(df_s['route_hash'].to_numpy()),
        torch.from_numpy(df_s['target_log_v'].to_numpy().astype(np.float32)),
    )
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, pin_memory=True, num_workers=2)

tr_loader = make_loader(tr, X_tr, bs=8192, shuffle=True)
va_loader = make_loader(va, X_va, bs=16384)
te_loader = make_loader(te, X_te, bs=16384)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepIndianBusETA(len(NUM_COLS)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=15)

for epoch in range(1, 16):
    model.train()
    tot = 0.0
    for xn, ci, cl, co, ro, y in tr_loader:
        xn, ci, cl, co, ro, y = xn.to(device), ci.to(device), cl.to(device), co.to(device), ro.to(device), y.to(device)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=device.type=='cuda'):
            pred = model(xn, ci, cl, co, ro)
            loss = pinball_loss(pred, y)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(y)
    sched.step()
    model.eval()
    vtot = 0.0
    with torch.no_grad():
        for xn, ci, cl, co, ro, y in va_loader:
            xn, ci, cl, co, ro, y = xn.to(device), ci.to(device), cl.to(device), co.to(device), ro.to(device), y.to(device)
            vtot += pinball_loss(model(xn, ci, cl, co, ro), y).item() * len(y)
    print(f'Epoch {epoch:02d} | train_pinball={tot/len(tr):.4f} | val_pinball={vtot/len(va):.4f}')

torch.save({'state_dict': model.state_dict(), 'mu': mu, 'sd': sd, 'num_cols': NUM_COLS}, OUT_DIR / 'deep_eta_a100.pt')
print('Saved A100 checkpoint to', OUT_DIR / 'deep_eta_a100.pt')